# Speculator Tests

Unified notebook for all speculator modes and test types.
Controlled by environment variables:
- `SPECULATOR_MODE`: DATA_ONLY (default), TRAIN_ONLY, ONLINE, OFFLINE
- `TEST_TYPE`: extraction (success path) or failure (failure scenarios)

Failure scenarios run sequentially — one job at a time to avoid GPU contention.

In [ ]:
import os
import warnings
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message=".*Unverified HTTPS.*")

from kubernetes import client as k8s_client
from kubeflow.trainer import TrainerClient
from kubeflow.common.types import KubernetesBackendConfig

openshift_api_url = os.getenv("OPENSHIFT_API_URL", "")
token = os.getenv("NOTEBOOK_USER_TOKEN", "")
namespace = os.getenv("NOTEBOOK_NAMESPACE", "default")
shared_pvc_name = os.getenv("SHARED_PVC_NAME", "shared-pvc")
speculator_mode = os.getenv("SPECULATOR_MODE", "DATA_ONLY")
test_type = os.getenv("TEST_TYPE", "extraction")

print(f"API: {openshift_api_url}")
print(f"Namespace: {namespace}")
print(f"PVC: {shared_pvc_name}")
print(f"Mode: {speculator_mode}")
print(f"Test type: {test_type}")

cfg = k8s_client.Configuration()
cfg.host = openshift_api_url
cfg.verify_ssl = False
cfg.api_key = {"authorization": f"Bearer {token}"}

api_client = k8s_client.ApiClient(cfg)
backend_cfg = KubernetesBackendConfig(client_configuration=api_client.configuration)
client = TrainerClient(backend_cfg)
print("TrainerClient initialized")

In [ ]:
training_runtime_name = os.getenv("TRAINING_RUNTIME")
if not training_runtime_name:
    raise RuntimeError("TRAINING_RUNTIME environment variable is required")

speculator_runtime = client.get_runtime(training_runtime_name)
if speculator_runtime is None:
    raise RuntimeError(f"Required runtime '{training_runtime_name}' not found")
print(f"Got runtime: {speculator_runtime.name}")

In [ ]:
# Model download — skipped entirely for failure tests (they test bad paths, not real models)
import os

notebook_pvc_path = "/opt/app-root/src"
model_local_path = f"{notebook_pvc_path}/models/Qwen3-1.7B"

if test_type == "failure":
    print("Skipping model download — failure tests do not need a real model")
else:
    s3_endpoint = os.getenv("AWS_DEFAULT_ENDPOINT", "")
    s3_access_key = os.getenv("AWS_ACCESS_KEY_ID", "")
    s3_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY", "")
    s3_bucket = os.getenv("AWS_STORAGE_BUCKET", "")
    model_s3_prefix = os.getenv("MODEL_S3_PREFIX", "models/Qwen3-1.7B")

    use_s3 = bool(s3_endpoint and s3_bucket and s3_access_key and s3_secret_key)

    if use_s3:
        print(f"S3 mode: downloading model to shared PVC")
        print(f"  Endpoint: {s3_endpoint}")
        print(f"  Bucket: {s3_bucket}")

        import s3fs

        endpoint_url = s3_endpoint if s3_endpoint.startswith("http") else f"https://{s3_endpoint}"
        fs = s3fs.S3FileSystem(
            key=s3_access_key,
            secret=s3_secret_key,
            endpoint_url=endpoint_url,
            use_ssl=endpoint_url.startswith("https"),
            config_kwargs={"signature_version": "s3v4"},
            client_kwargs={"verify": False},
        )

        if not os.path.exists(os.path.join(model_local_path, "config.json")):
            os.makedirs(model_local_path, exist_ok=True)
            remote_path = f"{s3_bucket}/{model_s3_prefix}"
            count = 0
            for remote_file in fs.find(remote_path):
                rel_path = remote_file[len(remote_path):].lstrip("/")
                if not rel_path:
                    continue
                local_file = os.path.join(model_local_path, rel_path)
                os.makedirs(os.path.dirname(local_file), exist_ok=True)
                fs.get(remote_file, local_file)
                count += 1
            print(f"  Downloaded {count} files to {model_local_path}")
        else:
            print(f"  Model already exists at {model_local_path}, skipping")

        print("S3 download complete")
    else:
        print("HuggingFace mode: downloading model from HF Hub")
        if not os.path.exists(os.path.join(model_local_path, "config.json")):
            from huggingface_hub import snapshot_download
            snapshot_download(
                repo_id="Qwen/Qwen3-1.7B",
                local_dir=model_local_path,
                token=os.getenv("HUGGINGFACE_HUB_TOKEN"),
                resume_download=True,
                local_dir_use_symlinks=False,
            )
            print(f"Model downloaded to {model_local_path}")
        else:
            print(f"Model already exists at {model_local_path}, skipping")

    print(f"Model ready at: {model_local_path}")

In [ ]:
import time

from kubeflow.trainer.rhai.speculator import (
    SpeculativeDecodingTrainer,
    SpeculatorMode,
    SpeculatorType,
    SpeculatorConfig,
)
from kubeflow.trainer.options.common import Name

FAILURE_EVENT_REASONS = {"BackOff", "CrashLoopBackOff", "Failed", "OOMKilled", "OOMKilling", "Killing"}


def check_job_failure(client, job_name):
    """Check if a TrainJob has failed using SDK APIs."""
    failure_confirmed = False
    details = []

    try:
        job = client.get_job(name=job_name)
        if job.status == "Failed":
            failure_confirmed = True
            details.append("job.status=Failed")
    except Exception as e:
        print(f"  get_job() error: {e}")

    try:
        events = client.get_job_events(name=job_name)
        for event in events:
            reason = getattr(event, "reason", "") or ""
            if reason in FAILURE_EVENT_REASONS:
                failure_confirmed = True
                message = getattr(event, "message", "") or ""
                details.append(f"event: reason={reason} message={message[:100]}")
    except Exception as e:
        print(f"  get_job_events() error: {e}")

    return failure_confirmed, details


def run_speculator_failure_scenario(client, scenario_name, trainer_kwargs, runtime,
                                    expect_sdk_error=False, expected_error=None):
    """Submit a speculator TrainJob expected to fail, verify the failure.

    If expect_sdk_error=True, the SDK should raise before creating a job.
    Otherwise, the job is created and should fail — verified via get_job()/get_job_events().
    Runs ONE job at a time and cleans up before returning.
    """
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_name}")
    if expect_sdk_error:
        print(f"Expected: SDK validation error (no job created)")
    else:
        print(f"Expected: Job failure detected via get_job()/get_job_events()")
    print(f"{'='*60}")

    job_name = None

    try:
        job_name = client.train(
            trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
            runtime=runtime,
        )

        if expect_sdk_error:
            print(f"FAILED: Expected SDK error but job was created: {job_name}")
            return False

        print(f"TrainJob created: {job_name}")

        # Wait for the job to start running or fail
        try:
            client.wait_for_job_status(name=job_name, status={"Running", "Failed"}, timeout=300)
        except Exception as e:
            print(f"Wait for Running/Failed raised: {e}")

        # Poll until failure is confirmed via SDK APIs
        failure_confirmed = False
        deadline = time.time() + 300

        while time.time() < deadline:
            failure_confirmed, details = check_job_failure(client, job_name)
            if failure_confirmed:
                print(f"Job failure confirmed via SDK: {'; '.join(details)}")
                break
            time.sleep(15)

        if not failure_confirmed:
            print(f"FAILED: Job failure not confirmed within timeout")
            return False

        print(f"PASSED: {scenario_name}")
        return True

    except Exception as e:
        if expect_sdk_error:
            error_msg = f"{type(e).__name__}: {e}"
            print(f"SDK error caught (expected): {error_msg}")
            if expected_error and expected_error not in error_msg:
                print(f"FAILED: Expected error containing '{expected_error}' but got: {error_msg}")
                return False
            print(f"PASSED: {scenario_name}")
            return True
        else:
            print(f"FAILED: {scenario_name} - unexpected error: {e}")
            return False

    finally:
        if job_name:
            try:
                client.delete_job(job_name)
                print(f"Deleted job: {job_name}")
            except Exception as e:
                print(f"Warning: failed to delete job {job_name}: {e}")


print("Helpers loaded")

In [ ]:
# ============================================================
# DATA_ONLY mode
# ============================================================

results = {}

if speculator_mode == "DATA_ONLY":
    vllm_gpu_count = int(os.getenv("VLLM_GPU_COUNT", "1"))
    vllm_resources = {"nvidia.com/gpu": vllm_gpu_count}

    job_name = os.getenv("JOB_NAME", "speculator-extract")
    default_model_uri = f"pvc://{shared_pvc_name}/models/Qwen3-1.7B"

    if test_type == "extraction":
        # --- Success path: submit extraction job, wait for completion ---
        trainer_kwargs = {
            "mode": SpeculatorMode.DATA_ONLY,
            "speculator_type": SpeculatorType.EAGLE3,
            "vllm_resources": vllm_resources,
        }

        verifier_model = os.getenv("VERIFIER_MODEL", default_model_uri)
        if verifier_model:
            trainer_kwargs["verifier_model"] = verifier_model

        dataset_name = os.getenv("DATASET_NAME", "")
        if dataset_name:
            trainer_kwargs["dataset_name"] = dataset_name

        output_dir = os.getenv("OUTPUT_DIR", "")
        if output_dir:
            trainer_kwargs["output_dir"] = output_dir

        max_samples = os.getenv("MAX_SAMPLES", "")
        if max_samples:
            trainer_kwargs["max_samples"] = int(max_samples)

        enable_progression = os.getenv("ENABLE_PROGRESSION_TRACKING", "")
        if enable_progression:
            trainer_kwargs["enable_progression_tracking"] = enable_progression.lower() == "true"

        regenerate_responses = os.getenv("REGENERATE_RESPONSES", "")
        if regenerate_responses and regenerate_responses.lower() == "true":
            trainer_kwargs["regenerate_responses"] = True

        config_overrides = {}
        target_layer_ids = os.getenv("TARGET_LAYER_IDS", "")
        if target_layer_ids:
            config_overrides["target_layer_ids"] = [int(x) for x in target_layer_ids.split(",")]
        datagen_concurrency = os.getenv("DATAGEN_CONCURRENCY", "")
        if datagen_concurrency:
            config_overrides["datagen_concurrency"] = int(datagen_concurrency)
        hidden_states_dtype = os.getenv("HIDDEN_STATES_DTYPE", "")
        if hidden_states_dtype:
            config_overrides["hidden_states_dtype"] = hidden_states_dtype
        if config_overrides:
            trainer_kwargs["config"] = SpeculatorConfig(**config_overrides)

        trainer_kwargs["packages_to_install"] = ["speculators==0.6.0", "torchvision==0.24.1"]

        print(f"Trainer kwargs: {trainer_kwargs}")

        try:
            submitted_name = client.train(
                trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
                runtime=speculator_runtime,
                options=[Name(name=job_name)],
            )
            print(f"TRAINJOB_NAME: {submitted_name}")

            client.wait_for_job_status(name=submitted_name, status={"Running"}, timeout=600)
            client.wait_for_job_status(name=submitted_name, status={"Complete", "Failed"}, timeout=3600)

            job = client.get_job(name=submitted_name)
            print(f"Job status: {job.status}")

            if os.getenv("TEST_IDEMPOTENCY", "").lower() == "true" and output_dir:
                idempotency_name = f"{job_name}-idempotency"
                print(f"Idempotency test: submitting second job '{idempotency_name}' to same output_dir")
                idempotency_submitted = client.train(
                    trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
                    runtime=speculator_runtime,
                    options=[Name(name=idempotency_name)],
                )
                print(f"IDEMPOTENCY_TRAINJOB_NAME: {idempotency_submitted}")

                client.wait_for_job_status(name=idempotency_submitted, status={"Running"}, timeout=600)
                client.wait_for_job_status(name=idempotency_submitted, status={"Complete", "Failed"}, timeout=3600)

                idempotency_job = client.get_job(name=idempotency_submitted)
                print(f"Idempotency job status: {idempotency_job.status}")

        except Exception as e:
            print(f"SDK_ERROR: {type(e).__name__}: {e}")

    elif test_type == "failure":
        # --- Failure scenarios: run sequentially, one job at a time ---
        print("Running DATA_ONLY failure scenarios...")

        # Scenario 1: Bad model path — job created but should fail at runtime
        results["Bad model path"] = run_speculator_failure_scenario(
            client=client,
            scenario_name="Bad model path",
            trainer_kwargs={
                "mode": SpeculatorMode.DATA_ONLY,
                "speculator_type": SpeculatorType.EAGLE3,
                "vllm_resources": vllm_resources,
                "verifier_model": f"pvc://{shared_pvc_name}/models/nonexistent-model",
                "dataset_name": "yahma/alpaca-cleaned",
                "output_dir": f"pvc://{shared_pvc_name}/speculator-output/fail",
                "config": SpeculatorConfig(target_layer_ids=[2, 14, 25, 28]),
                "packages_to_install": ["speculators==0.6.0", "torchvision==0.24.1"],
            },
            runtime=speculator_runtime,
            expect_sdk_error=False,
        )

else:
    print(f"Skipping — mode {speculator_mode} is not DATA_ONLY")

In [ ]:
# ============================================================
# TRAIN_ONLY mode
# ============================================================

if speculator_mode == "TRAIN_ONLY":
    results = {}

    train_gpu_count = int(os.getenv("TRAIN_GPU_COUNT", "1"))
    training_resources = {"nvidia.com/gpu": train_gpu_count}

    default_model_uri = f"pvc://{shared_pvc_name}/models/Qwen3-1.7B"
    verifier_model = os.getenv("VERIFIER_MODEL", default_model_uri)

    extract_output = os.getenv("OUTPUT_DIR", f"pvc://{shared_pvc_name}/speculator-output/extract")
    hidden_states_path = f"{extract_output}/hidden_states"
    data_path = extract_output
    train_output = os.getenv("TRAIN_OUTPUT_DIR", f"pvc://{shared_pvc_name}/speculator-output/train")

    target_layer_ids = None
    target_layer_ids_str = os.getenv("TARGET_LAYER_IDS", "")
    if target_layer_ids_str:
        target_layer_ids = [int(x) for x in target_layer_ids_str.split(",")]

    hidden_states_dtype = os.getenv("HIDDEN_STATES_DTYPE", "bfloat16")

    if test_type == "extraction":
        trainer_kwargs = {
            "mode": SpeculatorMode.TRAIN_ONLY,
            "speculator_type": SpeculatorType.EAGLE3,
            "training_resources": training_resources,
            "verifier_model": verifier_model,
            "hidden_states_path": hidden_states_path,
            "data_path": data_path,
            "output_dir": train_output,
            "epochs": 2,
            "lr": 1e-4,
            "packages_to_install": ["speculators==0.6.0", "torchvision==0.24.1"],
        }

        config_overrides = {
            "checkpoint_freq": 1.0,
        }
        if target_layer_ids:
            config_overrides["target_layer_ids"] = target_layer_ids
        if hidden_states_dtype:
            config_overrides["hidden_states_dtype"] = hidden_states_dtype
        trainer_kwargs["config"] = SpeculatorConfig(**config_overrides)

        print(f"TRAIN_ONLY trainer kwargs: {trainer_kwargs}")

        try:
            job1_name = client.train(
                trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
                runtime=speculator_runtime,
                options=[Name(name="speculator-train")],
            )
            print(f"TRAIN_ONLY TRAINJOB_NAME: {job1_name}")

            client.wait_for_job_status(name=job1_name, status={"Running"}, timeout=600)
            client.wait_for_job_status(name=job1_name, status={"Complete", "Failed"}, timeout=3600)

            job1 = client.get_job(name=job1_name)
            print(f"TRAIN_ONLY job status: {job1.status}")

            if job1.status != "Complete":
                raise RuntimeError(f"TRAIN_ONLY job failed with status: {job1.status}")

            # Checkpoint resume test: re-submit with resume_from_checkpoint=True
            print("Checkpoint resume: submitting new job with resume_from_checkpoint=True")

            resume_config = {
                "checkpoint_freq": 1.0,
                "resume_from_checkpoint": True,
            }
            if target_layer_ids:
                resume_config["target_layer_ids"] = target_layer_ids
            if hidden_states_dtype:
                resume_config["hidden_states_dtype"] = hidden_states_dtype
            trainer_kwargs["config"] = SpeculatorConfig(**resume_config)

            job2_name = client.train(
                trainer=SpeculativeDecodingTrainer(**trainer_kwargs),
                runtime=speculator_runtime,
                options=[Name(name="speculator-train-resume")],
            )
            print(f"RESUME TRAINJOB_NAME: {job2_name}")

            client.wait_for_job_status(name=job2_name, status={"Running"}, timeout=600)
            client.wait_for_job_status(name=job2_name, status={"Complete", "Failed"}, timeout=3600)

            job2 = client.get_job(name=job2_name)
            print(f"Resume job status: {job2.status}")

            if job2.status != "Complete":
                raise RuntimeError(f"Resume job failed with status: {job2.status}")

            print("TRAIN_ONLY extraction + checkpoint resume: PASSED")

        except Exception as e:
            print(f"SDK_ERROR: {type(e).__name__}: {e}")

    elif test_type == "failure":
        print("Running TRAIN_ONLY failure scenarios...")

        base_kwargs = {
            "mode": SpeculatorMode.TRAIN_ONLY,
            "speculator_type": SpeculatorType.EAGLE3,
            "training_resources": {"nvidia.com/gpu": 1},
            "verifier_model": f"pvc://{shared_pvc_name}/models/Qwen3-1.7B",
            "epochs": 1,
            "lr": 1e-4,
            "packages_to_install": ["speculators==0.6.0", "torchvision==0.24.1"],
        }
        if target_layer_ids:
            base_kwargs["config"] = SpeculatorConfig(target_layer_ids=target_layer_ids)

        # Scenario 1: Bad hidden_states_path — nonexistent path on PVC
        results["Bad hidden_states_path"] = run_speculator_failure_scenario(
            client=client,
            scenario_name="Bad hidden_states_path",
            trainer_kwargs={
                **base_kwargs,
                "hidden_states_path": f"pvc://{shared_pvc_name}/nonexistent/hidden_states",
                "data_path": f"pvc://{shared_pvc_name}/nonexistent/data",
                "output_dir": f"pvc://{shared_pvc_name}/speculator-output/fail-hs",
            },
            runtime=speculator_runtime,
            expect_sdk_error=False,
        )

        # Scenario 2: Bad data_path — nonexistent path on PVC
        results["Bad data_path"] = run_speculator_failure_scenario(
            client=client,
            scenario_name="Bad data_path",
            trainer_kwargs={
                **base_kwargs,
                "hidden_states_path": f"pvc://{shared_pvc_name}/nonexistent/hidden_states",
                "data_path": f"pvc://{shared_pvc_name}/nonexistent/data2",
                "output_dir": f"pvc://{shared_pvc_name}/speculator-output/fail-data",
            },
            runtime=speculator_runtime,
            expect_sdk_error=False,
        )

else:
    print(f"Skipping — mode {speculator_mode} is not TRAIN_ONLY")

In [ ]:
# ============================================================
# ONLINE mode (placeholder — to be implemented)
# ============================================================

if speculator_mode == "ONLINE":
    if test_type == "extraction":
        raise NotImplementedError("ONLINE extraction tests not yet implemented")
    elif test_type == "failure":
        raise NotImplementedError("ONLINE failure tests not yet implemented")
else:
    print(f"Skipping — mode {speculator_mode} is not ONLINE")

In [ ]:
# ============================================================
# OFFLINE mode (placeholder — to be implemented)
# ============================================================

if speculator_mode == "OFFLINE":
    if test_type == "extraction":
        raise NotImplementedError("OFFLINE extraction tests not yet implemented")
    elif test_type == "failure":
        raise NotImplementedError("OFFLINE failure tests not yet implemented")
else:
    print(f"Skipping — mode {speculator_mode} is not OFFLINE")

In [ ]:
# ============================================================
# Summary
# ============================================================

if test_type == "failure" and results:
    print(f"\n{'='*60}")
    print(f"SPECULATOR FAILURE SCENARIOS — {speculator_mode}")
    print(f"{'='*60}")
    for name, passed in results.items():
        status = "PASSED" if passed else "FAILED"
        print(f"  {name}: {status}")
    print(f"{'='*60}")

    if not all(results.values()):
        failed = [name for name, passed in results.items() if not passed]
        raise RuntimeError(f"Failed scenarios: {', '.join(failed)}")
    else:
        print("All scenarios passed")
else:
    print("Notebook execution completed")